# Manipulating databases directly

Tables hold data. Associated with eahc table in Spark is its relevant **metadata**, which is information about the table and its data: the schema, description, table name, database name, column names, partitions, physical location where the actual data resides, etc. All of this is stored in a central *metastore*.

Spark, by default, stores the metadata in `spark-warehouse` under the current folder.
However, we may change the default location by setting the Spark config variable `spark.sql.warehouse.dir` to another location, which can be set to a local or external distributed storage.


**Managed tables**: Spark manages both metadata and data. So, for example, dropping a table deletes the metadata and the data.

**Unmanaged tables**: Spark manages only meta data while we manage the data ourselves in an external data source. For example, dropping a table deletes the metadata but the not the external data.

In [120]:
from pyspark.sql import SparkSession, Row
from pyspark.sql.functions import *

In [121]:
spark = (SparkSession
        .builder
        .appName("Spark SQL Example 2")
        .getOrCreate())

In [122]:
csv_file = "flights/data/departuredelays.csv"

# Creating SQL Databases and Tables

In [123]:
spark.sql("CREATE DATABASE IF NOT EXISTS test")
spark.sql("USE test")

DataFrame[]

The data warehouse will be created in the current folder. To set a different location, use 

```spark.conf.set("spark.sql.warehouse.dir", "/path/to/warehouse")```

The following command tells Apache Spark to create native Spark data source tables (such as Parquet or Delta) by default instead of legacy Hive SerDe (TextFile) tables whenever we run a CREATE TABLE command that lacks an explicit USING or STORED AS clause. The native format is faster. Note that the metadata is still stored in Hive metastore.

In [124]:
spark.conf.set("spark.sql.legacy.createHiveTableByDefault", "false")

In [125]:
spark.sql("CREATE TABLE managed_us_delay_flights_tbl (date STRING, delay INT, \
distance INT,origin STRING, destination STRING)")

DataFrame[]

In [126]:
schema = "date STRING, delay INT,distance INT,origin STRING, destination STRING"

In [127]:
flights_df = spark.read.csv(csv_file, schema = schema)

The following is an alternate way to create a managed table (same as above, but using the DataFrame interface):
```flights_df.write.saveAsTable("managed_us_delay_flights_tbl")```

We can also created unmanaged tables from a data source such as CSV, Parquet, or JSON.

In [128]:
spark.sql("""CREATE TABLE local_us_delay_flights_tbl (date STRING, delay INT,distance INT,
origin STRING, destination STRING)
USING csv OPTIONS (LOCATION 'file:flights/data/departuredelays.csv')""")

DataFrame[]

# Creating Views

Spark can also create views on top of existing tables. Views
can be global (visible across all SparkSessions on a given cluster) or session-scoped
(visible only to a single SparkSession), and they are temporary: they disappear after
your Spark application terminates.

You can create a view from an existing table using SQL. For example, if you wish to
work on only the subset of the US flight delays data set with origin airports of New
York (JFK) and San Francisco (SFO), the following queries will create global temporary and temporary views consisting of just that slice of the table:

In [129]:
spark.sql("""
CREATE OR REPLACE GLOBAL TEMP VIEW us_origin_airport_SFO_global_tmp_view AS 
SELECT date, delay, origin, destination from managed_us_delay_flights_tbl WHERE
origin = 'SFO';
""")

DataFrame[]

In [130]:
spark.sql("""CREATE OR REPLACE TEMP VIEW us_origin_airport_JFK_tmp_view AS
SELECT date, delay, origin, destination from managed_us_delay_flights_tbl WHERE
origin = 'JFK'
""")

DataFrame[]

You can accomplish the same thing with the DataFrame API as follows

In [131]:
df_sfo = spark.sql("""SELECT date, delay, origin, destination FROM
managed_us_delay_flights_tbl WHERE origin = 'SFO'""")

In [132]:
df_jfk = spark.sql("""SELECT date, delay, origin, destination FROM
managed_us_delay_flights_tbl WHERE origin = 'JFK'""")

In [133]:
df_sfo.createOrReplaceGlobalTempView("us_origin_airport_SFO_global_tmp_view")
df_jfk.createOrReplaceTempView("us_origin_airport_JFK_tmp_view")

# Viewing Metadata

The Catalogis a high level abstraction for storing metadata.

In [134]:
spark.catalog.listDatabases()

[Database(name='default', catalog='spark_catalog', description='default database', locationUri='file:/home/amit/Documents/CS435-535-resources/examples/spark/spark-sql-python/spark-warehouse'),
 Database(name='test', catalog='spark_catalog', description='', locationUri='file:/home/amit/Documents/CS435-535-resources/examples/spark/spark-sql-python/spark-warehouse/test.db')]

In [135]:
spark.catalog.listTables()

[Table(name='local_us_delay_flights_tbl', catalog='spark_catalog', namespace=['test'], description=None, tableType='MANAGED', isTemporary=False),
 Table(name='managed_us_delay_flights_tbl', catalog='spark_catalog', namespace=['test'], description=None, tableType='MANAGED', isTemporary=False),
 Table(name='us_origin_airport_jfk_tmp_view', catalog=None, namespace=[], description=None, tableType='TEMPORARY', isTemporary=True)]

In [136]:
spark.catalog.listColumns("managed_us_delay_flights_tbl")

[Column(name='date', description=None, dataType='string', nullable=True, isPartition=False, isBucket=False, isCluster=False),
 Column(name='delay', description=None, dataType='int', nullable=True, isPartition=False, isBucket=False, isCluster=False),
 Column(name='distance', description=None, dataType='int', nullable=True, isPartition=False, isBucket=False, isCluster=False),
 Column(name='origin', description=None, dataType='string', nullable=True, isPartition=False, isBucket=False, isCluster=False),
 Column(name='destination', description=None, dataType='string', nullable=True, isPartition=False, isBucket=False, isCluster=False)]

# Caching SQL tables
Like DataFrames, we can cache and uncache SQL tables and views, We can also specify a table as LAZY, meaning that it should only be cached when it is first used instead of immediately:

-- In SQL
```
CACHE [LAZY] TABLE <table-name>
UNCACHE TABLE <table-name>
```


# Reading Tables into DataFrames
Often, data engineers build data pipelines as part of their regular data ingestion and
ETL processes. They populate Spark SQL databases and tables with cleansed data for
consumption by applications downstream.

Let’s assume we have an existing database, `test`, and table,
`managed_us_delay_flights_tbl`, ready for use. Instead of reading from an external JSON file, we can simply use SQL to query the table and assign the returned result to a
DataFrame:



In [137]:
us_flights_df = spark.sql("SELECT * FROM managed_us_delay_flights_tbl")
us_flights_df2 = spark.table("managed_us_delay_flights_tbl")

# Clean up

In [138]:
spark.sql("DROP TABLE IF EXISTS local_us_delay_flights_tbl")
spark.sql("DROP TABLE IF EXISTS managed_us_delay_flights_tbl")
spark.sql("DROP TABLE IF EXISTS df_us_delay_flights_tbl")

DataFrame[]

In [139]:
spark.sql("DROP DATABASE test")

DataFrame[]

In [140]:
spark.stop()